# Quick Start

See the [usage](../../../usage/data/), [design](../../../design/data/), and [development](../../../development/data/) documentation for more details on the Data stage.

## Imports

In [ ]:
# Import SuPAErnova
import supaernova

# Import pathlib, used to define paths required by SNPAE
from pathlib import Path

# Import Pretty Printing, only used for demonstration
from pprint import pp

## Configuration
To run the Data stage you need to provide SNPAE with a [DataStepConfig](../../../api/supaernova/configs/steps/data/#supaernova.configs.steps.data.DataStepConfig).

> **Warning**: These configuration options are likely to change in the future.

To begin, we will create a dictionary which will store our Data configurations.

In [ ]:
config = {}

config["data"] = {}

In addition to per-stage configuration, SNPAE also requires a number of global configurations which we will set here.

In [ ]:
# Set logging verbosity
# If True, then debug messages will be written to STDOUT
# These messages are written to the log files regardless, so you usually don't need to enable this
verbose = False

# Force SNPAE to rerun everything, even when it could normally reuse old results
# Usually only needed if you have changed your config after running it, however here we set it to True for demonstration purposes.
force = True

# Sets the base path to which all other paths are relative
# If base_path is relative, it is assumed to be relative to CWD
# Here we set it to the `examples/` directory
base_path = Path.cwd().parent

# Determines where all output (logs, checkpoints, plots, etc...) will go
out_path = base_path / "outputs" / "data" / "quick_start"

We now have our global configuration, and an empty Data stage configuration. Before we start configuring the Data stage, why don't we try running SNPAE:

In [ ]:
snpae = supaernova.prepare_config(
    config,
    verbose=verbose,
    force=force,
    base_path=base_path,
    out_path=out_path
)

As you can see, SNPAE does its best to warn you when your provided configuration has any problems. Here it's warning us that we're missing a number of required keys in our [DataStepConfig](../../../api/supaernova/configs/steps/data/#supaernova.configs.steps.data.DataStepConfig). Let's fill those in now.

### Required Settings

These are explained in detail in the [usage](../../../usage/data/) documentation. Note that the `data_dir` points to dummy data, so you will likely need to change it. Optional settings are explored in the [advanced](advanced.ipynb) example.

In [ ]:
config["data"] = {
    "data_dir": base_path.parent / "data",
    "meta": "meta.csv",
    "idr": "IDR_eTmax.txt",
    "mask": "mask_info_wmin_wmax.txt",
    "colourlaw": "colourlaws/F99_colourlaw.txt"
}

With everything fully configured, let's try running SNPAE again.

In [ ]:
snpae = supaernova.prepare_config(
    config,
    verbose=verbose,
    force=force,
    base_path=base_path,
    out_path=out_path
)
print("snpae:")
pp(snpae.model_dump())

As you can see, the `snpae` object has a lot more than the few keys we configured. Logging has been set up, relevant paths have been created, and a {{DataStep}} object has been created, ready to be run.

## Execution
Let's run our `snpae` object.

In [ ]:
snpae.run()
print("snpae:")
pp(snpae.model_dump())

As you can see, we now have a [DataStep](../../../api/supaernova/steps/data/#supaernova.steps.data.DataStep) object stored in `snpae.data_step`. This is here the results of the run are stored, ready to be used by later stages. The most important attributres are `snpae.data_step.data`, which contains the complete dataset, and `snpae.data_step.train_data` and `snpae.data_step.test_data` which are the training and testing kfolds respectively.

In [ ]:
print(f"data keys: {list(snpae.data_step.data.model_dump().keys())}")
print(f"Number of training kfolds: {len(snpae.data_step.train_data)}")
print(f"Number of SNe per training kfold: {[kfold.sn_name.shape[0] for kfold in snpae.data_step.train_data]}")
print(f"Number of testing kfolds: {len(snpae.data_step.test_data)}")
print(f"Number of SNe per testing kfold: {[kfold.sn_name.shape[0] for kfold in snpae.data_step.test_data]}")